In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
import json

In [18]:
with open("/content/drive/MyDrive/Μηχανική Μετάφραση_Εργασία 1/facebook_nllb_200_translations.json", "r", encoding = "utf-8") as f:
  fb_nllb = json.load(f)

with open("/content/drive/MyDrive/Μηχανική Μετάφραση_Εργασία 1/Helsinki_NLP_Translations.json", "r", encoding = "utf-8") as f:
  helsinki_nlp = json.load(f)

with open("/content/drive/MyDrive/Μηχανική Μετάφραση_Εργασία 1/Flores_ell.json", "r", encoding = "utf-8") as f:
  source = json.load(f)

with open("/content/drive/MyDrive/Μηχανική Μετάφραση_Εργασία 1/Flores_eng.json", "r", encoding = "utf-8") as f:
  ref = json.load(f)

In [19]:
import os
os.environ["USE_TF"] = "0"

In [7]:
!pip install unbabel-comet

  Using cached unbabel_comet-2.2.7-py3-none-any.whl.metadata (19 kB)
  Using cached entmax-1.3-py3-none-any.whl.metadata (348 bytes)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached jsonargparse-3.13.1-py3-none-any.whl.metadata (55 kB)
  Using cached numpy-1.26.4.tar.gz (15.8 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 7.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 10.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.7/529.7 kB 11.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 39.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 61.7 MB/s  0:00:00
  Created wheel for numpy: filename=numpy-1.26.4-cp313-cp313-linux_x86_64.whl size=7039801 sha256=bd9c37d5c2d19bd41bd4f6

In [1]:
import os

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

In [11]:
from huggingface_hub import login
login()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [12]:
from comet import download_model, load_from_checkpoint # Ο κώδικας για το commet από εδώ: https://github.com/Unbabel/COMET

In [14]:
model_path = download_model("Unbabel/wmt22-comet-da")

model = load_from_checkpoint(model_path)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

checkpoints/model.ckpt:   0%|          | 0.00/2.32G [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

hparams.yaml:   0%|          | 0.00/567 [00:00<?, ?B/s]

... (2 outputs left out)

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/core/saving.py:216: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


In [20]:
data_fb_nllb = [{"src": source, "mt": fb_nllb, "ref": ref}]


data_helsinki_nlp = [{"src": source, "mt": helsinki_nlp, "ref": ref}]


In [21]:
# Facebook NLLB
fb_nllb_output = model.predict(data_fb_nllb, batch_size=8, gpus=1)

# Helsinki NLP
helsinki_nlp_output = model.predict(data_helsinki_nlp, batch_size=8, gpus=1)

# COMET Scores
COMET_nllb = fb_nllb_output.system_score
COMET_helsinki = helsinki_nlp_output.system_score

print("Facebook NLLB COMET:", COMET_nllb)
print("Helsinki NLP COMET:", COMET_helsinki)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|

Facebook NLLB COMET: 0.8652027249336243
Helsinki NLP COMET: 0.8298905491828918
